In [1]:
# 환경 변수 로드
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="./env/.env")

api_key = os.getenv('OPENAI_API_KEY')

In [2]:
# Chat 모델 및 프롬프트 설정
from langchain.chat_models import ChatOpenAI
from langchain.storage import LocalFileStore
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import UnstructuredFileLoader
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(
    temperature= 0.1
)   

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True # 메시지 객체(리스트) 형태로 결과를 반환하도록 설정
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})

In [7]:
# 임베딩 캐시 저장소
cache_dir = LocalFileStore("./.cache/challenge_4/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size = 600,
    chunk_overlap = 100,
)

loader = UnstructuredFileLoader("./files/document.txt")
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

# 토큰 단위로 쪼개진 문서를 임베딩과정 거친 후 FAISS 벡터 스토어에 저장
vectorstore = FAISS.from_documents(docs, cached_embeddings)

# 1. Retriever: 질문과 관련된 문서 검색
retriever = vectorstore.as_retriever()

# 2. Prompt: 검색된 문서(context)와 질문(question)을 조합
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음의 문서를 참고해서 질문에 대답해. 만약 답을 모른다면 모른다고 말해.\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 검색된 문서 객체들을 하나의 문자열로 결합
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# 3. Chain 구성 (LCEL)
chain = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs, # 질문 → Retriever → 관련 문서 검색 → format_docs → context에 할당
        "question": lambda x: x["question"], # 질문 → 그대로 question에 할당
        "chat_history": lambda x: x["chat_history"] 
    } 
    | prompt 
    | llm
)

In [8]:
response1 = chain.invoke({
    "question" : "Aaronson 은 유죄인가요?",
    "chat_history": get_history()["chat_history"]
    })
add_message("Aaronson 은 유죄인가요?", response1.content)
print(response1.content)

Aaronson은 유죄로 판결받았습니다.


In [9]:
response2 = chain.invoke({
    "question" : "그가 테이블에 어떤 메시지를 썼나요?",
    "chat_history": get_history()["chat_history"]
    })
add_message("그가 테이블에 어떤 메시지를 썼나요?", response2.content)
print(response2.content)

그가 쓴 메시지는 "FREEDOM IS SLAVERY"와 "TWO AND TWO MAKE FIVE" 그리고 "GOD IS POWER"입니다.


In [10]:

response3 = chain.invoke({
    "question" : "Julia 는 누구인가요?",
    "chat_history": get_history()["chat_history"]
    })
add_message("Julia는 누구인가요?", response3.content)
print(response3.content)

Julia는 윈스턴과 사랑에 빠진 여성으로, 이야기에서 중요한 역할을 합니다. 그들은 빅 브라더의 권력에 대항하고자 하는 공통된 욕망을 공유하며 함께 모반을 일으키게 됩니다.
